### Phase 2: Data Preparation

In [1]:
import pandas as pd 
import numpy as np 
import warnings

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('../final_datasets/combined_datasets/era5_glaciers_datasets.csv')
df.shape

(13036, 70)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13036 entries, 0 to 13035
Data columns (total 70 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            13036 non-null  int64  
 1   glacier_id            13036 non-null  float64
 2   year                  13036 non-null  int64  
 3   glacier_name          13036 non-null  str    
 4   country_left          13036 non-null  str    
 5   region                13036 non-null  str    
 6   area_change           12847 non-null  float64
 7   elevation_change      12891 non-null  float64
 8   elevation_change_unc  12797 non-null  float64
 9   volume_change         12112 non-null  float64
 10  volume_change_unc     12106 non-null  float64
 11  index_right           13036 non-null  int64  
 12  rgi_id                13036 non-null  str    
 13  cenlon                13036 non-null  float64
 14  cenlat                13036 non-null  float64
 15  utm_zone              13036 no

In [4]:
df.describe()

,Unnamed: 0,glacier_id,year,area_change,elevation_change,elevation_change_unc,volume_change,volume_change_unc,index_right,cenlon,...,ssrd,strd,ro,es,sro,e,u10,v10,sp,tp
count,13036.000000,13036.000000,13036.000000,1.284700e+04,12891.000000,12797.000000,1.211200e+04,1.210600e+04,13036.000000,13036.000000,...,1.303600e+04,1.303600e+04,13036.000000,13036.000000,13036.000000,13036.000000,13036.000000,13036.000000,13036.000000,13036.000000
mean,6522.625499,65871.162933,2011.779994,1.214815e+06,-3.450149,4.546824,-3.480722e+06,3.469595e+06,3258.134474,84.295800,...,2.004194e+07,2.058994e+07,0.002860,-0.000076,0.001678,-0.000938,0.157905,0.063500,57100.868629,0.003950
std,3764.461738,22097.160113,6.028961,3.411525e+06,4.012567,1.876932,1.342450e+07,7.306259e+06,1836.748540,2.131604,...,1.479595e+06,1.773101e+06,0.001808,0.000037,0.001317,0.000371,0.288818,0.339151,3610.492856,0.001718
min,3.000000,904.000000,1952.000000,1.800000e+04,-175.400000,0.100000,-4.253925e+08,1.250000e+05,428.000000,80.810252,...,1.443644e+07,9.975764e+06,0.000193,-0.000206,0.000048,-0.002070,-0.401817,-0.441600,49889.045417,0.000924
25%,3262.750000,73731.000000,2009.000000,1.550000e+05,-4.805000,3.510000,-2.595000e+06,8.250000e+05,1770.750000,82.519025,...,1.899482e+07,1.949108e+07,0.001550,-0.000101,0.000755,-0.001183,-0.045667,-0.179411,54410.009750,0.002658
50%,6522.500000,74495.000000,2014.000000,3.730000e+05,-2.730000,4.180000,-8.700000e+05,1.530000e+06,2881.000000,83.931406,...,2.017458e+07,2.049751e+07,0.002465,-0.000074,0.001314,-0.000950,0.109870,-0.056364,57092.794500,0.003593
75%,9782.250000,75295.000000,2019.000000,9.590000e+05,-1.215000,5.292500,-2.650000e+05,3.189375e+06,4989.250000,86.499996,...,2.118343e+07,2.180212e+07,0.003805,-0.000048,0.002266,-0.000751,0.321154,0.230586,59284.454917,0.004904
max,13042.000000,216960.000000,2025.000000,6.105400e+07,30.017500,78.180000,6.113500e+07,1.348200e+08,6906.000000,88.176722,...,2.332857e+07,2.608420e+07,0.014379,0.000062,0.009601,-0.000019,1.606236,1.334671,70079.878333,0.014543


##### Step 1: handling duplicates

In [5]:
print(f"Before removing dupliate\n Shape : {df.shape}")
print(f"After removing dupliate\n Shape : {df.drop_duplicates().shape}")

Before removing dupliate
 Shape : (13036, 70)
After removing dupliate
 Shape : (13036, 70)


No duplicate rows present here

##### Step 2: handling missing values

In [6]:
# viewing missing percentages of each feature
df.isna().mean().mul(100).sort_values(ascending=False)

glac_name               97.852102
volume_change_unc        7.134090
volume_change            7.088064
elevation_change_unc     1.833384
area_change              1.449831
                          ...    
e                        0.000000
u10                      0.000000
v10                      0.000000
sp                       0.000000
tp                       0.000000
Length: 70, dtype: float64

Around ~98% of data are missing in `glac_name` and ~7.2% in `volumne_change_unc` and `volume_change`.

Now, removing `glac_name` because of larger number of missing values, but interploating other minor missing values

In [7]:
missing_percent = df.isna().mean() * 100

# missing values over 90%
missing_90_percent_cols = missing_percent[missing_percent > 90].index.to_list()
print(f"Features/Columns having more than 90% missing values: {missing_90_percent_cols}")


# missing values greater than 0% but less than 90%
missing_zero_percent_cols = missing_percent[(missing_percent > 0) & (missing_percent < 90)].index.to_list()
print(f"Features/Columns having > 0% and < 90% missing values: {missing_zero_percent_cols}")

Features/Columns having more than 90% missing values: ['glac_name']
Features/Columns having > 0% and < 90% missing values: ['area_change', 'elevation_change', 'elevation_change_unc', 'volume_change', 'volume_change_unc']


In [8]:
# since volume_change is the potential target variable, which must not be interploated so, dropping null values is better choice
missing_zero_percent_cols.remove("volume_change")
df.dropna(subset=['volume_change'], inplace=True)

In [9]:
print(f"After dropping ~7% missing values of 'volume_change': {df.shape}")

After dropping ~7% missing values of 'volume_change': (12112, 70)


In [10]:
df.drop(columns=missing_90_percent_cols, inplace=True)
df[missing_zero_percent_cols] = df[missing_zero_percent_cols].interpolate(method='linear', limit_direction='both')

# rechecking missing percentages|
df.isna().mean().mul(100).sort_values(ascending=False)


Unnamed: 0      0.0
glacier_id      0.0
year            0.0
glacier_name    0.0
country_left    0.0
               ... 
e               0.0
u10             0.0
v10             0.0
sp              0.0
tp              0.0
Length: 69, dtype: float64

None of the features have `NAN` (null) values

##### Step 3: handling unnecessary features

Now, **removing some unnecessary features** that have no roles such as unique identifiers (IDs), non-vital categorical columns like `region`.

In [11]:
df['primeclass'].value_counts()

primeclass
0    12112
Name: count, dtype: int64

`primeclass` has only one value i.e 0, which has no variance, so dropping it saves the memory consumption futher.

In [12]:
df['glacier_name'].value_counts()

glacier_name
LHOTSE NUP        5
NUPTSE            5
LHOTSE            5
MERA              5
RGI60-13.04765    4
                 ..
RGI60-15.03414    4
RGI60-15.03413    4
RGI60-15.03412    4
AX010             3
RGI60-15.04577    1
Name: count, Length: 3028, dtype: int64

In [13]:
categorical_cols = df.select_dtypes(include=['category', 'str']).columns.tolist()
print(categorical_cols)

['glacier_name', 'country_left', 'region', 'rgi_id', 'dem_source', 'country_right']


In [14]:
df['region'].value_counts()

region
15_south_asia_east    12088
13_central_asia          24
Name: count, dtype: int64

In [15]:
df['surge_type'].value_counts()

surge_type
0    12104
2        4
3        4
Name: count, dtype: int64

In [16]:
df['term_type'].value_counts()

term_type
9    12112
Name: count, dtype: int64

Since, region is heavily imbalanced as wen can see in the value which will be treated as a constant, so dropping it saves the memory consumption futher.

In [17]:
df['dem_source'].value_counts()

dem_source
COPDEM30    11932
COPDEM90      180
Name: count, dtype: int64

As `dem_source` is the name of the dataset that was used to compute the topography attributes. so dropping it is a wise decision.

In [18]:
print(df.columns.tolist())

['Unnamed: 0', 'glacier_id', 'year', 'glacier_name', 'country_left', 'region', 'area_change', 'elevation_change', 'elevation_change_unc', 'volume_change', 'volume_change_unc', 'index_right', 'rgi_id', 'cenlon', 'cenlat', 'utm_zone', 'area_km2', 'primeclass', 'conn_lvl', 'surge_type', 'term_type', 'zmin_m', 'zmax_m', 'zmed_m', 'zmean_m', 'slope_deg', 'aspect_deg', 'aspect_sec', 'dem_source', 'lmax_m', 'country_right', 'distance_degrees', 'd2m', 't2m', 'skt', 'stl1', 'stl2', 'stl3', 'stl4', 'lblt', 'licd', 'lict', 'lmld', 'lmlt', 'lshf', 'ltlt', 'asn', 'snowc', 'rsn', 'sde', 'sd', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'str', 'sshf', 'ssrd', 'strd', 'ro', 'es', 'sro', 'e', 'u10', 'v10', 'sp', 'tp']


In [19]:
unnecessary_cols = ["Unnamed: 0", "glacier_id", "rgi_id", "index_right", "primeclass", "country_left", "country_right", "utm_zone", "conn_lvl", "term_type"  ]

Since, `region`, `dem_sources` and other categorical columns except `glacier_name`

In [20]:
categorical_cols

['glacier_name',
 'country_left',
 'region',
 'rgi_id',
 'dem_source',
 'country_right']

In [21]:
cats_cols_without_glacier_name = ['country_left', 'region', 'rgi_id', 'dem_source', 'country_right']

In [22]:
cats_cols_without_glacier_name

['country_left', 'region', 'rgi_id', 'dem_source', 'country_right']

In [23]:
irrelevant_cols = unnecessary_cols + cats_cols_without_glacier_name
print(irrelevant_cols)
print(len(irrelevant_cols))

['Unnamed: 0', 'glacier_id', 'rgi_id', 'index_right', 'primeclass', 'country_left', 'country_right', 'utm_zone', 'conn_lvl', 'term_type', 'country_left', 'region', 'rgi_id', 'dem_source', 'country_right']
15


In [24]:
# dropping irrelevant features
df.drop(columns=irrelevant_cols, inplace=True)

print(f"Shape after dropping: {df.shape}")

Shape after dropping: (12112, 57)


##### Step 4: handling outliers

In [25]:
# getting numerical features
numerical_cols = df.select_dtypes(include=['int', 'float']).columns.tolist()
print(numerical_cols)

['year', 'area_change', 'elevation_change', 'elevation_change_unc', 'volume_change', 'volume_change_unc', 'cenlon', 'cenlat', 'area_km2', 'surge_type', 'zmin_m', 'zmax_m', 'zmed_m', 'zmean_m', 'slope_deg', 'aspect_deg', 'aspect_sec', 'lmax_m', 'distance_degrees', 'd2m', 't2m', 'skt', 'stl1', 'stl2', 'stl3', 'stl4', 'lblt', 'licd', 'lict', 'lmld', 'lmlt', 'lshf', 'ltlt', 'asn', 'snowc', 'rsn', 'sde', 'sd', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'str', 'sshf', 'ssrd', 'strd', 'ro', 'es', 'sro', 'e', 'u10', 'v10', 'sp', 'tp']


calculating outliers using IQR method

In [26]:
Q1 = df[numerical_cols].quantile(0.25)
Q3 = df[numerical_cols].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - (1.5 * IQR)
upper_limit = Q3 + (1.5 * IQR)

In [27]:
# print(f"Lower limit value: {lower_limit}")
# print(f"Upper limit value: {upper_limit}")

In [28]:
print("Dataset shape")
print(f"Before removing outliers: {df.shape}")

df_outliers_removal = df[((df[numerical_cols] >= lower_limit) & (df[numerical_cols] <= upper_limit)).all(axis=1)]

print(f"After removing outliers: {df_outliers_removal.shape}")

print(f"Reduced size: {((df.shape[0] - df_outliers_removal.shape[0] ) / df.shape[0]) * 100} %")

Dataset shape
Before removing outliers: (12112, 57)
After removing outliers: (5437, 57)
Reduced size: 55.11063408190224 %


If all outliers are dropped, then ~56.2% of data will be gone. Instead,

In [29]:
cols_to_clip = [col for col in numerical_cols if col not in ['cenlon', 'cenlat', 'year', 'volume_change']]

# capping which means replacing higher values than upper limit with upper limit value, and lower values than lower limit with lower limit value
df[cols_to_clip] = df[cols_to_clip].clip(lower=lower_limit, upper=upper_limit, axis=1)


print(f"Shape after clipping: {df.shape}") 


Shape after clipping: (12112, 57)


In [30]:
df['year']

36       1991
51       1996
59       1999
64       2002
65       2002
         ... 
12676    2019
12677    2019
12678    2019
12679    2019
12680    2019
Name: year, Length: 12112, dtype: int64

Deriving `acutal_mass_change` feature as a target variable from `volume_change`.

In [31]:
# average density of glacier ice (rho) = 850 kg/m^3
rho_ice = 850

# calculating and adding target variable
# Formula: Mass (Gt) = Volume (km^3) * (850 / 1000) taken from Huss (2013) "Density assumptions for converting geodetic glacier volume change to mass change"
df['actual_mass_change'] = df['volume_change'] * (rho_ice / 1000)    # unit is gigatones of water equivalent (g.w.e)


In [32]:
# dropping original volume features to prevent data leakage
df.drop(columns=['volume_change', 'volume_change_unc'], inplace=True)

In [33]:
df.set_index(['year', 'glacier_name'], inplace=True)

In [34]:
df

area_change  elevation_change  elevation_change_unc  \
year glacier_name                                                          
1991 AX010              568000.0           -8.6890                4.7900   
1996 AX010              495000.0           -5.5000                4.7900   
1999 AX010              450000.0           -2.4700                4.7900   
2002 LHOTSE NUP         375000.0           -2.2200                4.7900   
     NUPTSE             300000.0           -1.9700                4.7900   
...                          ...               ...                   ...   
2019 RGI60-15.06479     237000.0           -0.2425                5.3175   
     RGI60-15.06480     163000.0           -1.8925                6.0975   
     RGI60-15.06482    1036000.0           -9.9925                3.9025   
     RGI60-15.06483    1642000.0           -9.9925                3.6200   
     UNNAMED 25953      310000.0           -9.9925                4.5125   

                        cenlon     cenlat  area_km2  surge_type     zmin_m  \
year glacier_name                                                            
1991 AX010           86.555703  27.715657  0.426034           0  4977.4860   
1996 AX010           86.555703  27.715657  0.426034           0  4977.4860   
1999 AX010           86.555703  27.715657  0.426034           0  4977.4860   
2002 LHOTSE NUP      86.890141  27.936466  2.351926           0  4963.2207   
     NUPTSE          86.861770  27.938681  2.351926           0  4973.5070   
...                        ...        ...       ...         ...        ...   
2019 RGI60-15.06479  81.305940  30.034934  0.237507           0  4996.0845   
     RGI60-15.06480  81.313180  30.034130  0.163028           0  5543.9670   
     RGI60-15.06482  81.758096  30.206640  1.107028           0  5358.6480   
     RGI60-15.06483  81.775881  30.210528  1.924580           0  5346.4404   
     UNNAMED 25953   82.225038  29.716206  0.334456           0  4741.6577   

                          zmax_m     zmed_m  ...          strd        ro  \
year glacier_name                            ...                           
1991 AX010           5326.812500  5221.8740  ...  2.314026e+07  0.003685   
1996 AX010           5326.812500  5221.8740  ...  2.301200e+07  0.003965   
1999 AX010           5326.812500  5221.8740  ...  2.367911e+07  0.004271   
2002 LHOTSE NUP      7105.239513  5261.5150  ...  1.938723e+07  0.001766   
     NUPTSE          5957.843300  5359.0513  ...  1.938723e+07  0.001766   
...                          ...        ...  ...           ...       ...   
2019 RGI60-15.06479  5804.966000  5562.6533  ...  2.084211e+07  0.002760   
     RGI60-15.06480  5881.031700  5754.5650  ...  2.084211e+07  0.002760   
     RGI60-15.06482  5857.328000  5461.5674  ...  1.987127e+07  0.002933   
     RGI60-15.06483  6164.848600  5521.9014  ...  1.987127e+07  0.002933   
     UNNAMED 25953   5234.126500  5009.4126  ...  2.168697e+07  0.001725   

                           es       sro         e       u10       v10  \
year glacier_name                                                       
1991 AX010          -0.000035  0.002045 -0.001696  0.223243 -0.138062   
1996 AX010          -0.000023  0.002109 -0.001571  0.182166 -0.207591   
1999 AX010          -0.000011  0.002437 -0.001723  0.200245 -0.033395   
2002 LHOTSE NUP     -0.000061  0.000944 -0.000632  0.487552  0.008241   
     NUPTSE         -0.000061  0.000944 -0.000632  0.487552  0.008241   
...                       ...       ...       ...       ...       ...   
2019 RGI60-15.06479 -0.000081  0.001406 -0.000790  0.140719 -0.015267   
     RGI60-15.06480 -0.000081  0.001406 -0.000790  0.140719 -0.015267   
     RGI60-15.06482 -0.000109  0.002076 -0.000514 -0.311552  0.118888   
     RGI60-15.06483 -0.000109  0.002076 -0.000514 -0.311552  0.118888   
     UNNAMED 25953  -0.000029  0.001078 -0.001398 -0.109037 -0.164600   

                               sp        tp  actual_mass_change  
year glacier

In [35]:
# saving the clean datasets
df.to_parquet("../final_datasets/clean_datasets/1_data_preparation_dataset.parquet", engine="pyarrow")
# df.to_csv("../final_datasets/clean_datasets/1_data_preparation_dataset.csv")

In [41]:
df.info()

<class 'pandas.DataFrame'>
MultiIndex: 12112 entries, (np.int64(1991), 'AX010') to (np.int64(2019), 'UNNAMED 25953')
Data columns (total 54 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   area_change           12112 non-null  float64
 1   elevation_change      12112 non-null  float64
 2   elevation_change_unc  12112 non-null  float64
 3   cenlon                12112 non-null  float64
 4   cenlat                12112 non-null  float64
 5   area_km2              12112 non-null  float64
 6   surge_type            12112 non-null  int64  
 7   zmin_m                12112 non-null  float64
 8   zmax_m                12112 non-null  float64
 9   zmed_m                12112 non-null  float64
 10  zmean_m               12112 non-null  float64
 11  slope_deg             12112 non-null  float64
 12  aspect_deg            12112 non-null  float64
 13  aspect_sec            12112 non-null  int64  
 14  lmax_m                12112 n

##### Step 5: handling unit conversion

> Now, peforming **unit conversion**, since `area_change`, `volume_change`, `volume_change_unc` are in $km$ (kilometer) but other features like `elevation_change` in $m$ (meter)

In [36]:
# km2_cols = ['area_change']  # km^2
# km3_cols = ['volume_change', 'volume_change_unc']  # km^3
# kg_cols = ['rsn']  # kgm^3

Before Unit Conversion:

In [37]:
# df[km2_cols + km3_cols + kg_cols].head(8)

Unit Conversion: 

$km$ ==> $m$

$kg$ ==> $g$

Adding those 3 columns in meters unit. Later in feature engineering, these features in meters will be scaling and normalizing while the features in km and kg will be dropped.

In [38]:
# df['area_change_m'] = df[km2_cols] * 1000 * 1000  # 1km * 1km = 1000m * 1000m

# df[['volume_change_m', 'volume_change_unc_m']] = df[km3_cols] * 1000 * 1000 * 1000

# df['rsn_m'] = df[kg_cols] * 1000



In [39]:
# df.info()

After Unit conversion: 

In [40]:
# df[['area_change_m', 'volume_change_m', 'volume_change_unc_m', 'rsn_m']].head(8)